# 07 · A second training dataset (HttpParams) and two keyword fixes

CSIC comes from one shop in 2010, so its attacks are of a few kinds. [HttpParamsDataset](https://github.com/Morzeux/HttpParamsDataset) has ~31k labelled parameter values. Its attacks were made with other tools (sqlmap, XSSYA, Vega, FuzzDB), so it adds variety. Its normal values come from CSIC.

Each value is turned into a request `GET /?q=<value>`, the same way a search box would send it.

**Two keyword fixes, found on run3 misses (development data):**
- Command injection: also count a newline as a command separator, and a few more commands (`ifconfig`, `netstat`, `sleep 5` ...).
- SQL tautology: also match when brackets come before it, as in `OR ((1=1`.

We compare two models: **A** = CSIC only, **B** = CSIC + HttpParams train split. The HttpParams test split is never trained on.

In [1]:
import sys
from pathlib import Path
from urllib.parse import quote
import numpy as np
import pandas as pd
from sklearn.metrics import precision_score, recall_score

ROOT = Path.cwd() if (Path.cwd() / 'data').exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
from ml.detector import build_pipeline, has_payload, to_frame
from ml.train_detector import load_httpparams
from eval.rules import rule_predict
from eval.benign_tricky import rows as benign_rows
from eval.match_labels import match

csic = pd.read_csv(ROOT / 'data' / 'processed' / 'csic_clean.csv', keep_default_na=False)
Xc, yc = to_frame(csic['Method'], csic['path'], csic['body']), csic['label']
Xh, yh = load_httpparams(ROOT / 'data' / 'raw' / 'httpparams_train.csv')
hp_test = pd.read_csv(ROOT / 'data' / 'raw' / 'httpparams_test.csv', keep_default_na=False)
print(len(yc), 'CSIC,', len(yh), 'HttpParams train,', len(hp_test), 'HttpParams test')
hp_test.groupby('attack_type').size()

25608 CSIC, 20712 HttpParams train, 10355 HttpParams test


attack_type
cmdi                30
norm              6434
path-traversal      97
sqli              3617
xss                177
dtype: int64

In [2]:
def fit(X, y):
    keep = (y == 0) | has_payload(X)
    X, y = X[keep], y[keep]
    hand = [c for c in X.columns if c != 'text']
    return build_pipeline(hand, pos_weight=(y == 0).sum() / (y == 1).sum()).fit(X, y)

models = {
    'A: CSIC only': fit(Xc, yc),
    'B: CSIC + HttpParams': fit(pd.concat([Xc, Xh], ignore_index=True), pd.concat([yc, yh], ignore_index=True)),
}

## 1. Test sets

In [3]:
PAYLOAD_TYPES = ['sqli', 'xss', 'traversal', 'cmdi', '']
run3, _ = match(ROOT / 'data' / 'raw' / 'juiceshop_generated_run3.log', ROOT / 'data' / 'labels' / 'requests_run3.csv')
run3 = run3[run3['attack_type'].isin(PAYLOAD_TYPES)]   # brute force, scan, enum: session layer
hp = pd.DataFrame({'method': 'GET', 'path': ['/?q=' + quote(v, safe='') for v in hp_test['payload']], 'body': '',
                   'label': np.where(hp_test['label'] == 'anom', 'attack', 'normal'),
                   'attack_type': hp_test['attack_type'].replace('norm', '')})
sets = {'run3 (dev)': run3, 'benign tricky': benign_rows(), 'HttpParams test': hp}

def predict(name, d):
    if name == 'rules':
        return np.asarray(rule_predict(d['method'], d['path'], d['body']))
    return models[name].predict(to_frame(d['method'], d['path'], d['body']))

rows = []
for set_name, d in sets.items():
    y = (d['label'] != 'normal').values
    for name in ['rules', *models]:
        p = predict(name, d)
        rows.append({'test set': set_name, 'detector': name,
                     'recall': recall_score(y, p, zero_division=0) if y.any() else np.nan,
                     'false alarms': int((p[~y] == 1).sum()), 'normal requests': int((~y).sum())})
pd.DataFrame(rows).round(3)

,test set,detector,recall,false alarms,normal requests
0,run3 (dev),rules,0.803,0,3328
1,run3 (dev),A: CSIC only,0.990,0,3328
2,run3 (dev),B: CSIC + HttpParams,0.993,0,3328
3,benign tricky,rules,NaN,0,60
4,benign tricky,A: CSIC only,NaN,0,60
5,benign tricky,B: CSIC + HttpParams,NaN,0,60
6,HttpParams test,rules,0.930,1,6434
7,HttpParams test,A: CSIC only,0.969,1,6434
8,HttpParams test,B: CSIC + HttpParams,0.992,1,6434


## 2. Recall by attack type

In [4]:
tables = {}
for set_name in ['run3 (dev)', 'HttpParams test']:
    d = sets[set_name]
    y = (d['label'] != 'normal').values
    tables[set_name] = pd.DataFrame({name: pd.Series(predict(name, d)[y]).groupby(d['attack_type'].values[y]).mean()
                                     for name in ['rules', *models]}).round(3)
pd.concat(tables)

rules  A: CSIC only  B: CSIC + HttpParams
run3 (dev)      cmdi            0.969         0.990                 0.990
                sqli            0.829         0.974                 0.987
                traversal       0.293         1.000                 1.000
                xss             1.000         1.000                 1.000
HttpParams test cmdi            0.400         0.767                 0.767
                path-traversal  0.526         0.526                 0.928
                sqli            0.946         0.983                 0.996
                xss             0.910         0.966                 0.977

## 3. What this shows

- **Model A was trained only on CSIC, and it still finds about 97% of the HttpParams test attacks.** Those attacks were made by other tools, so this is a second piece of evidence that the model is not tied to its training data. The normal side is weaker evidence, because HttpParams' normal values come from CSIC.
- **Adding HttpParams (model B)** catches one more SQLi on run3, with no new false alarms. It also helps a lot on HttpParams path traversal. That part is expected, because B has seen the same kind of data.
- **Decision:** `ml/train_detector.py` now trains model B when `data/raw/httpparams_train.csv` exists.
- **Limit:** the keyword fixes and the choice of B were made on run3, so run3 numbers are optimistic. run2 can no longer test this model, because the features changed after it was used. The honest test is run4 and a third site.